Code to compute vapour pressure deficit (VPD) projections from daily tasmax and hursmin BARRA data.

Equations saturation vapour pressure http://www.bom.gov.au/climate/how/newproducts/images/IDCJHC02_notes.txt

Vapour pressure = exp (1.8096 + (17.269425 * Dew_Point)/(237.3 + Dew_Point))

Saturated Vapour pressure = exp (1.8096 + (17.269425 * Air_Temperature)/(237.3 + Air_Temperature))

Relative Humidity = Vapour pressure / Saturated vapour pressure * 100

Rearrange the formulae to get:

Vapour pressure = rh * 0.0061094 * exp((17.652 * t)/(243.04 + t))

A nice explainer about the relevance of VPD to fire: https://blog.ucsusa.org/carly-phillips/what-is-vapor-pressure-deficit-vpd-and-what-is-its-connection-to-wildfires/

In [1]:
import dask
from dask.distributed import Client, wait
from dask import delayed

client = Client(n_workers=7, threads_per_worker=1) 
#client = Client()

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 7,Total memory: 251.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44517,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46777,Total threads: 1
Dashboard: /proxy/42877/status,Memory: 35.88 GiB
Nanny: tcp://127.0.0.1:38369,


2026-10-01 14:14:14,280 - tornado.application - ERROR - Uncaught exception GET /individual-task-stream/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='gadi-cpu-bdw-0092.gadi.nci.org.au:54472', method='GET', uri='/individual-task-stream/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/tornado/web.py", line 3409, in wrapper
    return method(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("

In [2]:
#import all the stuff
from netCDF4 import Dataset
import xarray as xr
import numpy as np
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt
import glob
import sys
sys.path.append("/g/data/mn51/users/nb6195/project/vpd/VPD")
import VPD_functions as VPD

In [3]:
#function to compute VPD from tasmax and rh datasets
#input: are datasets of rh and tasmax
#output: dataset of daily vpd 

def vpd_calc(ds_rh, ds_tasmax):
    vpd = (1 - ds_rh/100) * 6.1094 * np.exp ((17.625 * ds_tasmax)/(243.04 + ds_tasmax)) #the formula that Blair uses from http://www.bom.gov.au/research/publications/cawcrreports/CTR_024.pdf

    vpd.attrs = {
        'long_name': 'Daily maximum vapour dressure deficit computed from tasmax and hursmin',
        'standard_name': 'vpd',
        'units': 'hPa',
        'acknowledgement' : "Development of data supported with funding from the Australian Climate Service."
#        'regrid_method': 'bilinear'
    }
    ds_vpd = xr.Dataset({'vpd' : vpd})
    ds_rh.close()
    ds_tasmax.close()

    return ds_vpd

In [4]:
#function to compute monthly VPD sum from daily values
#input: are datasets of rh and tasmax
#output: datasets of vpd and monthly_mean_vpd

def monthly_vpd_sum(ds_daily_vpd):
    monthly_vpd_sum = ds_daily_vpd.resample(time="1MS").sum(skipna=True)
    
    monthly_vpd_sum.attrs = {
        'long_name': 'Monthly cumulative vapour dressure deficit computed from BARRA tasmax and hursmin',
        'standard_name': 'monthly_vpd_sum',
        'units': 'hPa'
    }
    monthly_vpd_sum.to_dataarray().to_numpy()
    ds_monthly_vpd_sum = xr.Dataset({'monthly_vpd_sum' : monthly_vpd_sum})
    
    return ds_monthly_vpd_sum

In [5]:
#Set parameters
ddir = f"/g/data/ia39/australian-climate-service/release/CORDEX/output-CMIP6/DD/AUST-05i/BOM/ERA5/historical/hres/BARRAR2/v1/day/"
output_dir = '/g/data/mn51/users/nb6195/project/fire_indicators/data/'

In [6]:
# Bring in BARRA tasmax data
var1 = 'tasmax'
infiles=glob.glob(ddir+var1+f'/v20241216/{var1}_AUST-05i_ERA5_historical_hres_BOM_BARRAR2_v1_day_*.nc')
BARRA_tasmax = xr.open_mfdataset(infiles)[var1]

/jobfs/180256076.gadi-pbs/ipykernel_3958364/1320738960.py:4: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  BARRA_tasmax = xr.open_mfdataset(infiles)[var1]


In [7]:
# Bring in BARRA hursmin data
var2 = 'hursmin'
infiles=glob.glob(ddir+var2+f'/v20241216/{var2}_AUST-05i_ERA5_historical_hres_BOM_BARRAR2_v1_day_*.nc')
BARRA_hursmin = xr.open_mfdataset(infiles)[var2]

/jobfs/180256076.gadi-pbs/ipykernel_3958364/4229469996.py:4: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  BARRA_hursmin = xr.open_mfdataset(infiles)[var2]


In [8]:
#Create the datasets for vpd and monthly mean vpd
vpd = vpd_calc(BARRA_hursmin, BARRA_tasmax)

Create shorter versions for the required ML project

In [12]:
#vpd

In [10]:
vpd_cropped = vpd.sel(time=slice('1998-11-01T12:00:00.000000000', '2023-12-31T12:00:00.000000000'))

In [11]:
# Resample the data to Monthly Start ('1MS') and calculate the mean along the time dimension
# This converts daily data into chronological monthly steps (12 months * 25 years = 300 steps)
monthly_vpd = vpd_cropped.resample(time='1MS').mean(dim='time')

In [13]:
#monthly_vpd

In [14]:
# Save the processed data back to a new NetCDF file
#outfile = '/g/data/mn51/users/nb6195/project/fire_indicators/data/BARRA_monthly_average_vpd.nc'
#monthly_vpd.to_netcdf(outfile, engine='netcdf4')
#print(outfile)

In [15]:
# Calculate the 12-month rolling average
# 'min_periods=12' ensures we only get a value if a full year of prior data exists
rolling_vpd_12_month = monthly_vpd.rolling(time=12, center=False, min_periods=12).mean()

In [16]:
# Save the processed data back to a new NetCDF file
outfile = '/g/data/mn51/users/nb6195/project/fire_indicators/data/BARRA_12_monthly_average_vpd.nc'
rolling_vpd_12_month.to_netcdf(outfile, engine='netcdf4')
print(outfile)

2026-10-01 14:11:01,925 - distributed.worker - ERROR - Compute Failed
Key:       ('getitem-overlap-store-map-897dc3eecbb31208da357e841df2405b', 0, 0, 0)
State:     executing
Task:  <Task ('getitem-overlap-store-map-897dc3eecbb31208da357e841df2405b', 0, 0, 0) _execute_subgraph(...)>
Exception: "ValueError('Moving window (=12) must between 1 and 11, inclusive')"
Traceback: ''



ValueError: Moving window (=12) must between 1 and 11, inclusive

In [17]:
rolling_vpd_3_month = monthly_vpd.rolling(time=3, center=False, min_periods=3).mean()

In [18]:
outfile = '/g/data/mn51/users/nb6195/project/fire_indicators/data/BARRA_vpd_3month_mean.nc'
rolling_vpd_3_month.to_netcdf(outfile, engine='netcdf4')
print(outfile)

2026-10-01 14:59:42,799 - distributed.worker - ERROR - Compute Failed
Key:       ('getitem-overlap-store-map-d680d3cf8bf5adbe19368e6100a4b9cd', 0, 0, 0)
State:     executing
Task:  <Task ('getitem-overlap-store-map-d680d3cf8bf5adbe19368e6100a4b9cd', 0, 0, 0) _execute_subgraph(...)>
Exception: "ValueError('Moving window (=3) must between 1 and 2, inclusive')"
Traceback: ''



ValueError: Moving window (=3) must between 1 and 2, inclusive

## Calculate monthly sums

In [ ]:
ds_monthly_vpd_sum = monthly_vpd_sum(vpd_cropped)

In [ ]:
ds_monthly_vpd_sum

In [ ]:
#print vpd to an external file
outfile = '/g/data/mn51/users/nb6195/project/fire_indicators/data/BARRA_monthly_vpd_sum.nc'
ds_monthly_vpd_sum.to_netcdf(outfile, engine='netcdf4')
print(outfile)

## Calculate 12 monthly sums

In [ ]:
rolling_12m_sum_vpd = monthly_vpd_sum.rolling(time=12, min_periods=12).sum()

In [ ]:
#print vpd to an external file
outfile = '/g/data/mn51/users/nb6195/project/fire_indicators/data/BARRA_12monthly_vpd_sum.nc'
rolling_12m_sum_vpd.to_netcdf(outfile, engine='netcdf4')
print(outfile)

In [ ]:
#print monthly mean ds to external file

outfile = '/g/data/mn51/users/nb6195/project/fire_indicators/data/BARRA_monthly_mean_vpd.nc'
monthly_mean_vpd.to_netcdf(outfile, engine='netcdf4')
print(outfile)